# Lecture 18 — Same task, four commands: clean / empty / wrong / tr × 10 init states (T4)

**First:** Runtime type **T4 GPU** (Kaggle: Accelerator **GPU T4 x2**, internet on; `/content` → `/kaggle/working`). If Lecture 12's runtime is still open you can skip cells 1–5 and continue from 6 (the original LIBERO must be installed; NOT the LIBERO-Plus runtime — Lecture 16 warning).

Flow: (1) hardware → (2) EGL → (3) VLANeXt + LIBERO → (4) Lecture 12 patch → (5) checkpoint → **(6) Lecture 18 patch** → (7) conditions + config → (8) a single run: 1 model load + 40 episodes → (9) analysis: table, `episodes.csv`, `runs.csv` → (10) videos.

Time (hypothesis, T4): install ~10 min + download ~5 min + model load ~3 min + 40 episodes ~30–60 min (a failed episode runs up to 220 steps; the `empty`/`tr` conditions may take longer). Write the times you measured into `projects/05-…/README.md`.

In [ ]:
# 1) Hardware: cc (7, 5) = T4
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch, psutil, shutil
print("torch", torch.__version__, "| GPU", torch.cuda.get_device_name(0), "| cc", torch.cuda.get_device_capability(0))
print("RAM %.1f GB | disk free %.0f GB" % (psutil.virtual_memory().total/1e9, shutil.disk_usage("/").free/1e9))

In [ ]:
# 2) MuJoCo EGL (VLANeXt COMMON_ISSUES #1)
!apt-get -qq update && apt-get -qq install -y libgl1-mesa-dev libegl1-mesa-dev libxrandr2 libxcursor1 libxinerama1 libxrender1 > /dev/null
import os; os.environ["MUJOCO_GL"] = "egl"; os.environ["PYOPENGL_PLATFORM"] = "egl"; print("ok")

In [ ]:
# 3) VLANeXt + original LIBERO (same as Lecture 12 cell 3)
%cd /content
!git clone -q https://github.com/DravenALG/VLANeXt && echo "clone ok"
%cd /content/VLANeXt
!git log -1 --format='VLANeXt %h %ci'
!mkdir -p third_party && git clone -q https://github.com/Lifelong-Robot-Learning/LIBERO.git third_party/LIBERO
!pip install -q "transformers==5.5.0" "diffusers==0.36.0" accelerate huggingface_hub sentencepiece h5py opencv-python "imageio[ffmpeg]" \
    "robosuite==1.4.1" "mujoco==3.8.1" bddl easydict cloudpickle gym pyyaml wandb tqdm tensorflow-datasets rlds scipy 2>&1 | tail -2
!pip install -q ./third_party/LIBERO 2>&1 | tail -1
!printf 'N\n' | python -c "import libero.libero; print('LIBERO ok')"

In [ ]:
# 4) Lecture 12 patch (sdpa, mmap, VLANEXT_TASK_IDS) — BEFORE the Lecture 18 patch
!curl -sO https://raw.githubusercontent.com/mesutd0nmez/vla-robot-learning-tr/main/code/day12/vlanext_t4_patch.py
!python vlanext_t4_patch.py /content/VLANeXt

In [ ]:
# 5) Checkpoint (16.4 GB; the same file as in Lecture 12/16)
!hf download DravenALG/VLANeXt VLANeXt_libero_spatial.pt --local-dir checkpoints && ls -lh checkpoints

In [ ]:
# 6) Lecture 18 patch: condition = EpisodeSpec field; init state × condition; condition= in the log; condition in the video name; RUN_TAG
!curl -sO https://raw.githubusercontent.com/mesutd0nmez/vla-robot-learning-tr/main/code/day18/vlanext_cond_patch.py
!python vlanext_cond_patch.py /content/VLANeXt
# Expected: 10 "[+] ... applied" (first time) — if you see "ERROR: ... expected line not found" the repo has changed; switch to Plan B in cell 8b.

In [ ]:
# 7) Conditions (JSON produced from datasets/01) + config. clean = null → the eval code puts in task.language itself.
!curl -sO https://raw.githubusercontent.com/mesutd0nmez/vla-robot-learning-tr/main/code/day18/conditions_task0.json && curl -s -o config/t4_cond_config.yaml https://raw.githubusercontent.com/mesutd0nmez/vla-robot-learning-tr/main/code/day18/t4_cond_config.yaml
import json; CONDS = open("conditions_task0.json", encoding="utf-8").read(); print(json.dumps(json.loads(CONDS), ensure_ascii=False, indent=2))
# If you are on Kaggle (29 GB RAM): num_parallel_envs 2 → 4, so that the four conditions of the same init state run in the same batch
# !sed -i 's/num_parallel_envs: 2/num_parallel_envs: 4/' config/t4_cond_config.yaml

In [ ]:
# 8) A SINGLE run: task 0, 10 init states × 4 conditions = 40 episodes, one model load. (~35–65 min, hypothesis)
import os, time
os.environ["VLANEXT_TASK_IDS"] = "0"
os.environ["VLANEXT_CONDITIONS"] = CONDS
os.environ["VLANEXT_RUN_TAG"] = "_4cond"
os.environ["PYTHONPATH"] = "/content/VLANeXt/third_party/LIBERO"
os.environ["MUJOCO_EGL_DEVICE_ID"] = "0"
os.environ.pop("LIBERO_CONFIG_PATH", None); os.environ.pop("VLANEXT_INSTR_OVERRIDE", None)
t0 = time.time()
!CUDA_VISIBLE_DEVICES=0 python -m scripts.libero_bench_eval --config config/t4_cond_config.yaml 2>&1 | grep -v Warning | grep -E "Conditions|Eval setup|Episode result|Eval finished|Error|error" | tee eval_stdout_4cond.txt
print("total time %.1f min" % ((time.time() - t0) / 60))

### 8b) Plan B — only if the Lecture 18 patch could not be applied
Use Lecture 16's patch (`VLANEXT_INSTR_OVERRIDE`) and do four separate runs (4 model loads; same `seed`, same 10 init states):
```python
!curl -sO https://raw.githubusercontent.com/mesutd0nmez/vla-robot-learning-tr/main/code/day16/vlanext_plus_patch.py && python vlanext_plus_patch.py /content/VLANeXt
import subprocess, glob
for cond, text in [("clean", None), ("empty", ""), ("wrong", json.loads(CONDS)["wrong"]), ("tr", json.loads(CONDS)["tr"])]:
    env = dict(os.environ, VLANEXT_TASK_IDS="0", VLANEXT_RUN_TAG="_" + cond); env.pop("VLANEXT_CONDITIONS", None); env.pop("VLANEXT_INSTR_OVERRIDE", None)
    if text is not None: env["VLANEXT_INSTR_OVERRIDE"] = text
    subprocess.run(["python", "-m", "scripts.libero_bench_eval", "--config", "config/t4_cond_config.yaml"], env=env, stdout=open(f"stdout_{cond}.txt", "w"), stderr=subprocess.STDOUT)
    print(cond, sorted(glob.glob(f"checkpoints/libero_spatial_checkpoint_*_libero_{cond}*/eval.log")))
```
Note: the Lecture 16 patch adds RUN_TAG to `libero_plus_bench_eval.py`, not to `libero_bench_eval.py`; for RUN_TAG you need item 5 of the Lecture 18 patch — if that could not be applied either, rename the folder `checkpoints/libero_spatial_checkpoint_*_libero_SR*` by hand after each run. Analysis: `python analyze_4cond.py --logs clean=... empty=... wrong=... tr=... --out results/`.

In [ ]:
# 9) Analysis: table (k/n, Wilson 95%, discordant pairs, McNemar / Fisher p) + episodes.csv + runs.csv
!curl -sO https://raw.githubusercontent.com/mesutd0nmez/vla-robot-learning-tr/main/code/day18/analyze_4cond.py
import glob
logs = sorted(glob.glob("checkpoints/libero_spatial_checkpoint_*_libero_4cond*/eval.log"), key=os.path.getmtime)
print(logs)
!python analyze_4cond.py {logs[-1]} --run-id vlanext_sp0_4cond_01 --out results/
!cat results/episodes.csv | head -12

In [ ]:
# 10) Videos: watch one failed episode from each condition — which bowl does the gripper go to? ('faithful' or 'biased' in the wrong condition?)
from IPython.display import Video, display
vids = sorted(glob.glob("checkpoints/libero_spatial_checkpoint_*_libero_4cond*/*.mp4"))
for cond in ["clean", "empty", "wrong", "tr"]:
    fails = [v for v in vids if f"task={cond}__" in v and "success=False" in v]
    print(cond, "failed:", len(fails), "/", len([v for v in vids if f"task={cond}__" in v]))
    if fails: display(Video(fails[0], embed=True, width=320))

## Next
- `results/` (episodes.csv, runs.csv, table.md), `eval.log` and one failed video per condition → `projects/05-spatial0-4kosul/` (create it from the template; in the README put the cell durations, the mean time per episode, and in how many videos the gripper went to the other bowl in the `wrong` condition).
- The 4 rows of `runs.csv` follow the repo convention (`run_id, model, checkpoint, suite, task_id, lang, condition, n_episodes, success_rate`).
- Lecture 19: we will read this table — which pattern (figure 07), is there a shortcut, what do the Wilson intervals say, is it worth going up to 20/50 episodes.